# Grokking: Figure 1 reproduction

This notebook is a self-contained reproduction of the left panel of Figure 1 in Power et al., [Grokking: Generalization Beyond Overfitting on Small Algorithmic Datasets](https://arxiv.org/abs/2201.02177).

It trains a small decoder-only transformer on division modulo 97, using a fixed random 50/50 train-validation split. The paper-specific run uses Adam with no weight decay for 1,000,000 optimization steps and repeats the experiment with three model seeds. Training accuracy should approach 100% long before validation accuracy does.

Run the cells in order. The final training cell is intentionally expensive: three million optimizer updates can take hours even on a GPU and substantially longer on CPU.

In [1]:
from __future__ import annotations

import math
import random
import time
from dataclasses import dataclass, replace

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

plt.style.use("seaborn-v0_8-whitegrid")
torch.set_float32_matmul_precision("high")

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if getattr(torch.backends, "mps", None) is not None
    and torch.backends.mps.is_available()
    else "cpu"
)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@dataclass(frozen=True)
class Figure1Config:
    prime: int = 97
    train_fraction: float = 0.5
    d_model: int = 128
    n_heads: int = 4
    n_layers: int = 2
    mlp_mult: int = 4
    batch_size: int = 512
    learning_rate: float = 1e-3
    beta1: float = 0.9
    beta2: float = 0.98
    warmup_steps: int = 10
    max_steps: int = 1_000_000
    number_of_seeds: int = 3
    evaluation_points: int = 250


CONFIG = Figure1Config()
print(f"PyTorch {torch.__version__}; device = {DEVICE}")
CONFIG

PyTorch 2.14.0; device = mps


/Users/ashwin/Code/mech_interp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Figure1Config(prime=97, train_fraction=0.5, d_model=128, n_heads=4, n_layers=2, mlp_mult=4, batch_size=512, learning_rate=0.001, beta1=0.9, beta2=0.98, warmup_steps=10, max_steps=1000000, number_of_seeds=3, evaluation_points=250)

## Dataset and tokenization

There are 97 choices for the numerator and 96 nonzero choices for the denominator, giving 9,312 equations. Division is multiplication by the denominator's modular inverse.

Following the authors' released implementation, an equation is tokenized as

    <eos> x / y = answer

and teacher-forced targets are

    answer <eos>

Loss and equation-level accuracy are calculated only on those two right-hand-side targets. The vocabulary is shared with the paper's other arithmetic and permutation tasks; retaining that output vocabulary avoids making this task artificially easier. A fixed NumPy seed shuffles the equations once, after which the first half is training data and the second half is validation data.

In [2]:
PAPER_OPERATORS = sorted(
    [
        "+", "-", "*", "/", "**2+", "**3+", "+*", "+-",
        "(x._value//y)if(y._value%2==1)else(x-y)_mod_97",
        "copy", "reverse", "s5", "s5aba", "s5conj", "sort",
        "x**2+y**2_mod_97", "x**2+y**2+x*y_mod_97",
        "x**2+y**2+x*y+x_mod_97", "x**3+x*y_mod_97",
        "x**3+x*y**2+y_mod_97",
    ]
)

EOS_TOKEN = 0
EQUALS_TOKEN = 1
OPERATOR_TOKEN = 2 + PAPER_OPERATORS.index("/")
NUMBER_OFFSET = 2 + len(PAPER_OPERATORS)
VOCAB_SIZE = NUMBER_OFFSET + CONFIG.prime + math.factorial(5)


def division_mod_prime(x: int, y: int, prime: int) -> int:
    if y == 0:
        raise ValueError("Zero has no multiplicative inverse.")
    return (x * pow(y, -1, prime)) % prime


def build_figure1_datasets(
    config: Figure1Config,
) -> tuple[TensorDataset, TensorDataset]:
    equations: list[list[int]] = []
    targets: list[list[int]] = []

    # Match the released dataset's pre-shuffle order: answer first,
    # denominator second, then derive the numerator = denominator * answer.
    for answer in range(config.prime):
        for denominator in range(1, config.prime):
            numerator = (denominator * answer) % config.prime
            assert division_mod_prime(numerator, denominator, config.prime) == answer

            answer_token = NUMBER_OFFSET + answer
            equations.append(
                [
                    EOS_TOKEN,
                    NUMBER_OFFSET + numerator,
                    OPERATOR_TOKEN,
                    NUMBER_OFFSET + denominator,
                    EQUALS_TOKEN,
                    answer_token,
                ]
            )
            targets.append([answer_token, EOS_TOKEN])

    permutation = np.random.RandomState(0).permutation(len(equations))
    inputs = torch.tensor(np.asarray(equations)[permutation], dtype=torch.long)
    labels = torch.tensor(np.asarray(targets)[permutation], dtype=torch.long)

    split_index = round(config.train_fraction * len(inputs))
    return (
        TensorDataset(inputs[:split_index], labels[:split_index]),
        TensorDataset(inputs[split_index:], labels[split_index:]),
    )


def decode_equation(tokens: torch.Tensor) -> str:
    x = int(tokens[1]) - NUMBER_OFFSET
    y = int(tokens[3]) - NUMBER_OFFSET
    answer = int(tokens[5]) - NUMBER_OFFSET
    return f"{x} / {y} = {answer} (mod {CONFIG.prime})"


TRAIN_DATASET, VAL_DATASET = build_figure1_datasets(CONFIG)

assert len(TRAIN_DATASET) == len(VAL_DATASET) == 4_656
assert not set(map(tuple, TRAIN_DATASET.tensors[0].tolist())) & set(
    map(tuple, VAL_DATASET.tensors[0].tolist())
)

print(f"vocabulary size: {VOCAB_SIZE}")
print(f"training equations: {len(TRAIN_DATASET):,}")
print(f"validation equations: {len(VAL_DATASET):,}")
for index in range(5):
    print(" ", decode_equation(TRAIN_DATASET[index][0]))

vocabulary size: 239
training equations: 4,656
validation equations: 4,656
  77 / 47 = 78 (mod 97)
  11 / 23 = 30 (mod 97)
  18 / 3 = 6 (mod 97)
  62 / 82 = 67 (mod 97)
  92 / 25 = 58 (mod 97)


## Model

The network is a causal, decoder-only transformer with two post-normalized blocks, width 128, four attention heads, a 4× ReLU MLP, sinusoidal position encodings, and no dropout. It reads the six-token teacher-forced equation and emits predictions after the equals sign and after the answer token.

In [3]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int) -> None:
        super().__init__()
        if d_model % n_heads:
            raise ValueError("d_model must be divisible by n_heads")

        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.query = nn.ModuleList(
            nn.Linear(d_model, self.d_head, bias=False) for _ in range(n_heads)
        )
        self.key = nn.ModuleList(
            nn.Linear(d_model, self.d_head, bias=False) for _ in range(n_heads)
        )
        self.value = nn.ModuleList(
            nn.Linear(d_model, self.d_head, bias=False) for _ in range(n_heads)
        )
        self.output = nn.Linear(d_model, d_model, bias=False)

    def forward(self, hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        heads = []
        for query, key, value in zip(self.query, self.key, self.value):
            q, k, v = query(hidden), key(hidden), value(hidden)
            scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)
            scores = scores.masked_fill(~mask, float("-inf"))
            heads.append(torch.softmax(scores, dim=-1) @ v)
        return self.output(torch.cat(heads, dim=-1))


class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, mlp_mult: int) -> None:
        super().__init__()
        self.attention = CausalSelfAttention(d_model, n_heads)
        self.attention_norm = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_mult * d_model, bias=False),
            nn.ReLU(),
            nn.Linear(mlp_mult * d_model, d_model, bias=False),
        )
        self.mlp_norm = nn.LayerNorm(d_model)

    def forward(self, hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        hidden = self.attention_norm(hidden + self.attention(hidden, mask))
        return self.mlp_norm(hidden + self.mlp(hidden))


class GrokkingTransformer(nn.Module):
    def __init__(self, config: Figure1Config) -> None:
        super().__init__()
        if config.d_model % 2:
            raise ValueError("Sinusoidal positions require an even d_model")

        self.embedding = nn.Embedding(VOCAB_SIZE, config.d_model)

        positions = torch.arange(6, dtype=torch.float32).unsqueeze(1)
        dimensions = torch.arange(0, config.d_model, 2, dtype=torch.float32)
        angles = positions / (10_000 ** (dimensions / config.d_model))
        position_encoding = torch.empty(6, config.d_model)
        position_encoding[:, 0::2] = torch.sin(angles)
        position_encoding[:, 1::2] = torch.cos(angles)
        self.register_buffer("position_encoding", position_encoding, persistent=False)

        self.blocks = nn.ModuleList(
            DecoderBlock(config.d_model, config.n_heads, config.mlp_mult)
            for _ in range(config.n_layers)
        )
        self.unembedding = nn.Linear(config.d_model, VOCAB_SIZE, bias=False)
        self.register_buffer(
            "causal_mask",
            torch.ones(6, 6, dtype=torch.bool).tril(),
            persistent=False,
        )

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        sequence_length = tokens.size(1)
        hidden = self.embedding(tokens)
        hidden = hidden + self.position_encoding[:sequence_length]

        mask = self.causal_mask[:sequence_length, :sequence_length]
        for block in self.blocks:
            hidden = block(hidden, mask)

        # Position -2 predicts the answer; position -1 predicts trailing EOS.
        return self.unembedding(hidden[:, -2:, :])


preview_model = GrokkingTransformer(CONFIG)
preview_logits = preview_model(TRAIN_DATASET.tensors[0][:4])
print(f"logit shape: {tuple(preview_logits.shape)}")
print(f"parameters: {sum(p.numel() for p in preview_model.parameters()):,}")

logit shape: (4, 2, 239)
parameters: 455,424


## Training and evaluation

Each reported accuracy is equation-level: both the answer and trailing end token must be correct. Evaluation occurs at logarithmically spaced steps so the plot resolves both early memorization and late generalization without repeatedly evaluating at all one million updates.

In [4]:
@torch.inference_mode()
def evaluate(
    model: nn.Module,
    dataset: TensorDataset,
    device: torch.device,
) -> dict[str, float]:
    model.eval()
    loader = DataLoader(dataset, batch_size=min(2048, len(dataset)), shuffle=False)
    loss_function = nn.CrossEntropyLoss(reduction="sum")

    total_loss = 0.0
    total_correct = 0
    total_equations = 0

    for tokens, targets in loader:
        tokens, targets = tokens.to(device), targets.to(device)
        logits = model(tokens)
        total_loss += float(
            loss_function(logits.flatten(0, 1), targets.flatten()).item()
        )
        predictions = logits.argmax(dim=-1)
        total_correct += int((predictions == targets).all(dim=-1).sum().item())
        total_equations += targets.size(0)

    return {
        "loss": total_loss / (2 * total_equations),
        "accuracy": total_correct / total_equations,
    }


def make_evaluation_steps(max_steps: int, number_of_points: int) -> set[int]:
    steps = np.geomspace(1, max_steps, num=number_of_points)
    return set(np.unique(np.rint(steps).astype(int)).tolist()) | {1, max_steps}


def first_step_at_accuracy(
    history: list[dict[str, float]],
    split: str,
    threshold: float = 0.99,
) -> int | None:
    for row in history:
        if row[f"{split}_accuracy"] >= threshold:
            return int(row["step"])
    return None


def train_one_seed(
    config: Figure1Config,
    model_seed: int,
) -> tuple[list[dict[str, float]], GrokkingTransformer]:
    seed_everything(model_seed)

    loader_generator = torch.Generator().manual_seed(model_seed)
    train_loader = DataLoader(
        TRAIN_DATASET,
        batch_size=min(config.batch_size, math.ceil(len(TRAIN_DATASET) / 2)),
        shuffle=True,
        generator=loader_generator,
        drop_last=False,
    )
    loader_iterator = iter(train_loader)

    model = GrokkingTransformer(config).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.learning_rate,
        betas=(config.beta1, config.beta2),
        weight_decay=0.0,
    )
    loss_function = nn.CrossEntropyLoss()
    evaluation_steps = make_evaluation_steps(
        config.max_steps, config.evaluation_points
    )
    history: list[dict[str, float]] = []

    progress = tqdm(
        range(1, config.max_steps + 1),
        desc=f"seed {model_seed}",
        unit="step",
        mininterval=1.0,
    )
    for step in progress:
        try:
            tokens, targets = next(loader_iterator)
        except StopIteration:
            loader_iterator = iter(train_loader)
            tokens, targets = next(loader_iterator)

        # Linear warmup over the first ten updates, then a constant 1e-3.
        learning_rate = config.learning_rate * min(
            step / config.warmup_steps, 1.0
        )
        for parameter_group in optimizer.param_groups:
            parameter_group["lr"] = learning_rate

        model.train()
        tokens, targets = tokens.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(tokens)
        loss = loss_function(logits.flatten(0, 1), targets.flatten())
        loss.backward()
        optimizer.step()

        if step in evaluation_steps:
            train_metrics = evaluate(model, TRAIN_DATASET, DEVICE)
            validation_metrics = evaluate(model, VAL_DATASET, DEVICE)
            row = {
                "step": step,
                "train_loss": train_metrics["loss"],
                "train_accuracy": train_metrics["accuracy"],
                "validation_loss": validation_metrics["loss"],
                "validation_accuracy": validation_metrics["accuracy"],
            }
            history.append(row)
            progress.set_postfix(
                train=f"{row['train_accuracy']:.3f}",
                validation=f"{row['validation_accuracy']:.3f}",
            )

    return history, model


def summarize(history: list[dict[str, float]], model_seed: int) -> None:
    final = history[-1]
    print(
        f"seed {model_seed}: "
        f"train={final['train_accuracy']:.3%}, "
        f"validation={final['validation_accuracy']:.3%}, "
        f"train>=99% at {first_step_at_accuracy(history, 'train')}, "
        f"validation>=99% at {first_step_at_accuracy(history, 'validation')}"
    )

In [5]:
# This is the paper-scale experiment: 3 seeds x 1,000,000 updates.
# For a quick plumbing check, first try:
# run_config = replace(CONFIG, max_steps=20, number_of_seeds=1, evaluation_points=10)
run_config = CONFIG

histories: list[list[dict[str, float]]] = []
trained_models: list[GrokkingTransformer] = []

started_at = time.perf_counter()
for model_seed in range(run_config.number_of_seeds):
    history, model = train_one_seed(run_config, model_seed)
    histories.append(history)
    trained_models.append(model)
    summarize(history, model_seed)

print(f"total training time: {(time.perf_counter() - started_at) / 3600:.2f} hours")

seed 0:   4%|▎     | 43482/1000000 [26:02<9:32:53, 27.83step/s, train=0.996, validation=0.020]


KeyboardInterrupt: 

In [ ]:
def plot_figure1_left(
    histories: list[list[dict[str, float]]],
) -> tuple[plt.Figure, plt.Axes]:
    figure, axis = plt.subplots(figsize=(6.4, 4.1))

    for seed_index, history in enumerate(histories):
        steps = [row["step"] for row in history]
        axis.plot(
            steps,
            [100 * row["train_accuracy"] for row in history],
            color="#ff0000",
            alpha=0.7,
            linewidth=1.25,
            label="training" if seed_index == 0 else None,
        )
        axis.plot(
            steps,
            [100 * row["validation_accuracy"] for row in history],
            color="#008000",
            alpha=0.7,
            linewidth=1.25,
            label="validation" if seed_index == 0 else None,
        )

    axis.set(
        xscale="log",
        xlim=(1, run_config.max_steps),
        ylim=(-2, 102),
        title="Modular division (training on 50% of data)",
        xlabel="Optimization steps",
        ylabel="Accuracy (%)",
    )
    axis.legend()
    figure.tight_layout()
    return figure, axis


figure, axis = plot_figure1_left(histories)
plt.show()